In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
import matplotlib.pyplot as plt
import seaborn as sns
from cuml.cluster import SpectralClustering
from sklearn.decomposition import PCA

class Clustering(object):

    def __init__(self, n_clusters: int, n_components: str, random_state: str,
                 n_neighbors: int, n_init: int, eigen_tol: str, affinity: str,
                 verbose: bool, output_type: str,
                 plotting_x: str, plotting_y: str,):
        self.n_clusters = n_clusters
        self.n_components = n_components
        self.random_state = random_state
        self.n_neighbors = n_neighbors
        self.n_init = n_init
        self.eigen_tol = eigen_tol
        self.affinity = affinity
        self.verbose = verbose
        self.output_type = output_type
        self.plotting_x = plotting_x
        self.plotting_y = plotting_y

    def df_read(self):
        global df_pandas_global
        
        df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],
                                date_format = {'Date': '%d/%m/%Y', 'Time': '%H:%M:%S'},
                                dayfirst = True)
        df_pandas_global = df_pandas
    
    def to_cudf(self):
        global df_cudf_global
        
        df_cudf = cudf.DataFrame(df_pandas_global).copy()
        df_cudf_global = df_cudf
        
    def converting(self):
        global df_cudf_converted_global
        
        df_cudf_converted_global = df_cudf_global.astype(float).copy()

    def SpectralClustering(self):
        global SpectralClustering_global
        global labels_global
        global cluster_centers_global

        SpectralClustering_float = cuml.cluster.SpectralClustering(n_clusters=self.n_clusters, n_components=self.n_components, 
                                                                   random_state=self.random_state, 
                                                                   n_neighbors=self.n_neighbors, 
                                                                   n_init=self.n_init, 
                                                                   eigen_tol=self.eigen_tol, 
                                                                   affinity=self.affinity, 
                                                                   verbose=self.verbose, output_type=self.output_type)
        SpectralClustering = SpectralClustering_float.fit(df_cudf_converted_global)
        SpectralClustering_global = SpectralClustering

        labels = SpectralClustering_float.labels_
        labels_global = cudf.DataFrame(labels, columns=['Labels'])

    def labels_concat(self):
        global df_concat_global
        global df_concat_pandas_global

        df_concat = cudf.concat([df_cudf_converted_global, labels_global], axis=1)
        df_concat_global = df_concat

        df_concat_pandas = df_concat.to_pandas()
        df_concat_pandas_global = df_concat_pandas

    def plotting(self):
        
        sns.scatterplot(data=df_concat_pandas_global, x=self.plotting_x, y=self.plotting_y, hue='Labels', palette='Set1')
        plt.show()
        
    def main(self):
        self.df_read()
        self.to_cudf()
        self.converting()
        st = time.time()
        self.SpectralClustering()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.labels_concat()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.plotting()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        